# 03 — Training: Model Rekomendasi Latihan (XGBoost + Optuna)

**Stack:**
- **XGBoost 2.x** sebagai primary model (replace RandomForest)
- **Optuna 3.5+** untuk hyperparameter tuning (TPE sampler, 50 trial)
- **5-fold StratifiedKFold** CV
- Train **2 model terpisah** (workout_type + intensity) untuk fleksibilitas

**Baseline comparison:** RandomForest (untuk evidence XGBoost lebih baik)

**Target:**
- F1-macro workout_type ≥ 0.85 (per EightGym Indonesia 2024)
- F1-macro intensity ≥ 0.80

**Output:**
- `output/models/workout_xgb_workout_type.pkl`
- `output/models/workout_xgb_intensity.pkl`
- `output/models/optuna_study_workout.pkl`
- `output/models/baseline_rf_comparison.json`

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import time
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

np.random.seed(42)
os.makedirs('output/models', exist_ok=True)

# Load data
X_train_bal = pd.read_parquet('output/preprocessed/X_train_balanced.parquet')
yw_train_bal = pd.read_parquet('output/preprocessed/yw_train_balanced.parquet')['workout_type_label']
X_test_scaled = pd.read_parquet('output/preprocessed/X_test_scaled.parquet')
yw_test = pd.read_parquet('output/preprocessed/yw_test.parquet')['workout_type_label']

X_train_bal_int = pd.read_parquet('output/preprocessed/X_train_balanced_intensity.parquet')
yi_train_bal = pd.read_parquet('output/preprocessed/yi_train_balanced.parquet')['intensity_label']
yi_test = pd.read_parquet('output/preprocessed/yi_test.parquet')['intensity_label']

print(f'Workout train (balanced): {X_train_bal.shape}, classes: {yw_train_bal.nunique()}')
print(f'Intensity train (balanced): {X_train_bal_int.shape}, classes: {yi_train_bal.nunique()}')

In [ ]:
# ── 1. Baseline: RandomForest (untuk comparison) ──
print('=== BASELINE: RandomForest ===')
rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_bal, yw_train_bal)
rf_pred = rf.predict(X_test_scaled)
rf_acc = accuracy_score(yw_test, rf_pred)
rf_f1 = f1_score(yw_test, rf_pred, average='macro', zero_division=0)
print(f'RF — Accuracy: {rf_acc:.4f} | F1-macro: {rf_f1:.4f}')

In [ ]:
# ── 2. XGBoost dengan Optuna TPE ──
print('=== OPTUNA TUNING: XGBoost (Workout Type) ===')

def objective_workout(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 100, 600, step=50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2),
    }
    model = XGBClassifier(
        **params,
        random_state=42,
        n_jobs=-1,
        eval_metric='mlogloss',
        tree_method='hist'  # Faster
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_bal, yw_train_bal,
                              cv=cv, scoring='f1_macro', n_jobs=-1)
    return scores.mean()

start = time.time()
study_workout = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    study_name='workout_xgb'
)
# 50 trial — bisa diturunkan ke 20 kalau lambat
study_workout.optimize(objective_workout, n_trials=50, show_progress_bar=True)
elapsed = time.time() - start

print(f'\n✅ Tuning workout selesai dalam {elapsed:.1f} detik')
print(f'Best CV F1-macro: {study_workout.best_value:.4f}')
print(f'Best params: {study_workout.best_params}')

In [ ]:
# ── 3. Train Final Workout Model dengan best params ──
best_params = study_workout.best_params

model_workout = XGBClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss',
    tree_method='hist'
)
model_workout.fit(X_train_bal, yw_train_bal)

# Test set evaluation
xgb_pred = model_workout.predict(X_test_scaled)
xgb_acc = accuracy_score(yw_test, xgb_pred)
xgb_f1 = f1_score(yw_test, xgb_pred, average='macro', zero_division=0)

print('=== HASIL XGBoost Workout Type ===')
print(f'Accuracy: {xgb_acc:.4f}')
print(f'F1-macro: {xgb_f1:.4f}')
print(f'\nKomparasi dengan baseline RF:')
print(f'  RF      — Acc: {rf_acc:.4f} | F1: {rf_f1:.4f}')
print(f'  XGBoost — Acc: {xgb_acc:.4f} | F1: {xgb_f1:.4f}')
print(f'  Improvement: {(xgb_f1 - rf_f1)*100:+.2f} pp F1-macro')

print(f'\n{"✅" if xgb_f1 >= 0.85 else "⚠️"} Target F1 ≥ 0.85: {"PASS" if xgb_f1 >= 0.85 else "NEEDS TUNING"}')

In [ ]:
# ── 4. Train Intensity Model (sama pattern, fewer trials) ──
print('=== OPTUNA TUNING: XGBoost (Intensity) ===')

def objective_intensity(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    model = XGBClassifier(**params, random_state=42, n_jobs=-1,
                          eval_metric='mlogloss', tree_method='hist')
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(model, X_train_bal_int, yi_train_bal,
                           cv=cv, scoring='f1_macro', n_jobs=-1).mean()

study_intensity = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_intensity.optimize(objective_intensity, n_trials=30, show_progress_bar=True)

model_intensity = XGBClassifier(
    **study_intensity.best_params,
    random_state=42, n_jobs=-1, eval_metric='mlogloss', tree_method='hist'
)
# Adjust intensity labels (1,2,3 → 0,1,2 untuk XGBoost)
yi_train_xgb = yi_train_bal - 1
yi_test_xgb = yi_test - 1
model_intensity.fit(X_train_bal_int, yi_train_xgb)

int_pred = model_intensity.predict(X_test_scaled) + 1  # back to 1,2,3
int_acc = accuracy_score(yi_test, int_pred)
int_f1 = f1_score(yi_test, int_pred, average='macro', zero_division=0)
print(f'\n=== HASIL XGBoost Intensity ===')
print(f'Accuracy: {int_acc:.4f} | F1-macro: {int_f1:.4f}')

In [ ]:
# ── 5. Plots: Feature Importance + Optuna History ──
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Feature importance
fi = model_workout.feature_importances_
fi_df = pd.DataFrame({'feature': X_train_bal.columns, 'importance': fi})
fi_df = fi_df.sort_values('importance', ascending=True).tail(15)
axes[0].barh(fi_df['feature'], fi_df['importance'], color='steelblue')
axes[0].set_title('Top-15 Feature Importance (Workout XGBoost)', fontweight='bold')
axes[0].set_xlabel('Importance')

# Optuna history
trials_df = study_workout.trials_dataframe()
axes[1].plot(trials_df['number'], trials_df['value'], marker='o', alpha=0.5)
axes[1].axhline(study_workout.best_value, color='red', linestyle='--',
                label=f'Best: {study_workout.best_value:.4f}')
axes[1].set_title('Optuna Optimization History', fontweight='bold')
axes[1].set_xlabel('Trial')
axes[1].set_ylabel('F1-macro (CV)')
axes[1].legend()

plt.tight_layout()
plt.savefig('output/models/training_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6. Save Models & Studies ──
joblib.dump(model_workout, 'output/models/workout_xgb_workout_type.pkl', compress=3)
joblib.dump(model_intensity, 'output/models/workout_xgb_intensity.pkl', compress=3)
joblib.dump(study_workout, 'output/models/optuna_study_workout.pkl', compress=3)
joblib.dump(study_intensity, 'output/models/optuna_study_intensity.pkl', compress=3)

# Save baseline comparison
comparison = {
    'baseline_rf': {
        'accuracy': round(float(rf_acc), 4),
        'f1_macro': round(float(rf_f1), 4),
    },
    'xgboost_workout_type': {
        'accuracy': round(float(xgb_acc), 4),
        'f1_macro': round(float(xgb_f1), 4),
        'cv_f1_macro': round(float(study_workout.best_value), 4),
        'best_params': study_workout.best_params,
        'n_trials': len(study_workout.trials),
    },
    'xgboost_intensity': {
        'accuracy': round(float(int_acc), 4),
        'f1_macro': round(float(int_f1), 4),
        'cv_f1_macro': round(float(study_intensity.best_value), 4),
        'best_params': study_intensity.best_params,
    },
    'improvement_pp': {
        'xgb_vs_rf_f1': round(float((xgb_f1 - rf_f1) * 100), 2),
    },
    'target_met': {
        'f1_workout_ge_0.85': bool(xgb_f1 >= 0.85),
        'f1_intensity_ge_0.80': bool(int_f1 >= 0.80),
    },
}
with open('output/models/training_metrics.json', 'w') as f:
    json.dump(comparison, f, indent=2)

print('✅ All models saved.')
for f in sorted(os.listdir('output/models')):
    size = os.path.getsize(f'output/models/{f}') / 1024
    print(f'  output/models/{f:45s} ({size:.1f} KB)')

print('\n=== FINAL METRICS ===')
print(json.dumps(comparison, indent=2))